In [ ]:
# OTHELLO bootstrap: make the package importable from notebooks/
import sys
from pathlib import Path
_repo_root = Path.cwd().parent.resolve()
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))


In [4]:
import ast
import pandas as pd
import json



In [10]:
names = ['qald', 'mintaka','hotpot']

dataframes = {name:pd.read_csv(f'../data/sparql/{name}_sparql_results.csv') for name in names}

In [2]:
def parse_wikidata(result):
    try:
        data = ast.literal_eval(result)
    except (ValueError, SyntaxError, TypeError):
        return None

    # Handle boolean results (e.g., ASK queries)
    if 'boolean' in data.get('head', {}):
        return str(data['boolean'])
    
    bindings = data.get('results', {}).get('bindings', [])
    if not bindings:
        return None

    # Try each binding until we find a value
    for dict_result in bindings:
        if not isinstance(dict_result, dict):
            continue

        # Priority 1: Look for *Label fields (most common for entity names)
        label_keys = [k for k in dict_result.keys() if k.endswith('Label')]
        for key in label_keys:
            value = dict_result.get(key, {}).get('value')
            if value:
                return value
        
        # Priority 2: Look for date/time fields
        date_keys = [k for k in dict_result.keys() if any(x in k.lower() for x in ['date', 'time', 'inception'])]
        for key in date_keys:
            value = dict_result.get(key, {}).get('value')
            if value:
                # Clean up ISO datetime format if needed
                return value.split('T')[0] if 'T' in str(value) else value
        
        # Priority 3: Look for numeric fields (year, age, population, etc.)
        for key in dict_result.keys():
            if key not in label_keys:  # Skip labels we already checked
                value = dict_result.get(key, {}).get('value')
                if value:
                    return value
    
    return None

In [7]:
from pprint import pprint

In [10]:
jsn = "{'head': {'vars': ['timeZone', 'timeZoneLabel']}, 'results': {'bindings': [{'timeZone': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q2212'}, 'timeZoneLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'UTC−07:00'}}, {'timeZone': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q3134980'}, 'timeZoneLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'Mountain Time Zone'}}]}}"

parsed = parse_wikidata(jsn)

In [17]:
jsn2 = json.loads(jsn)

In [15]:
jsn = '{"head":{"vars":["timeZone","timeZoneLabel"]},"results":{"bindings":[{"timeZone":{"type":"uri","value":"http://www.wikidata.org/entity/Q2212"},"timeZoneLabel":{"xml:lang":"en","type":"literal","value":"UTC−07:00"}},{"timeZone":{"type":"uri","value":"http://www.wikidata.org/entity/Q3134980"},"timeZoneLabel":{"xml:lang":"en","type":"literal","value":"Mountain Time Zone"}}]}}'


In [18]:
pprint(jsn2)

{'head': {'vars': ['timeZone', 'timeZoneLabel']},
 'results': {'bindings': [{'timeZone': {'type': 'uri',
                                        'value': 'http://www.wikidata.org/entity/Q2212'},
                           'timeZoneLabel': {'type': 'literal',
                                             'value': 'UTC−07:00',
                                             'xml:lang': 'en'}},
                          {'timeZone': {'type': 'uri',
                                        'value': 'http://www.wikidata.org/entity/Q3134980'},
                           'timeZoneLabel': {'type': 'literal',
                                             'value': 'Mountain Time Zone',
                                             'xml:lang': 'en'}}]}}


In [ ]:

for name, data in dataframes.items():
    print(f'starting processing for {name}')
    names = []
    for index, result in enumerate(data['Results']):
        parsed_name = parse_wikidata(result)
        names.append(parsed_name)
        print(f'Processed row {index + 1}', parsed_name)

    data['names'] = names

    path = f'../data/processed_names/{name}_processed.csv'
    
    data.to_csv(path)
    print(f'{data} saved to {path}')

starting processing for qald
Processed row 1 UTC−07:00
Processed row 2 Marcus Junius Brutus
Processed row 3 Q136722681
Processed row 4 John F. Kennedy
Processed row 5 Idaho
Processed row 6 Ralf Gessler
Processed row 7 journalist
Processed row 8 50033
Processed row 9 None
Processed row 10 Andorra
Processed row 11 None
Processed row 12 None
Processed row 13 None
Processed row 14 None
Processed row 15 None
Processed row 16 None
Processed row 17 4
Processed row 18 None
Processed row 19 None
Processed row 20 None
Processed row 21 None
Processed row 22 None
Processed row 23 taxon
Processed row 24 None
Processed row 25 Neil Gaiman
Processed row 26 Rembrandt
Processed row 27 None
Processed row 28 1995-01-01T00:00:00Z
Processed row 29 None
Processed row 30 Edwin Catmull
Processed row 31 None
Processed row 32 Elfriede Jelinek
Processed row 33 None
Processed row 34 None
Processed row 35 None
Processed row 36 Yangtze
Processed row 37 None
Processed row 38 None
Processed row 39 None
Processed row 4

: 